In [4]:
# ================================
# 1. LOAD DATASET
# ================================

import pandas as pd   # needed to read CSV

df = pd.read_csv("data/dataset.csv")

print("Dataset:")
print(df.head())


# ================================
# 2. PREPROCESS DATA
# ================================

from sklearn.preprocessing import LabelEncoder   # needed for encoding
from sklearn.model_selection import train_test_split  # needed for splitting

# separate features and label
X = df.drop("disease", axis=1)
y = df["disease"]

# encode labels
le = LabelEncoder()
y = le.fit_transform(y)

print("\nEncoded Labels:", y)


# ================================
# 3. SPLIT DATA
# ================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


# ================================
# 4. CREATE & TRAIN MODEL
# ================================

from xgboost import XGBClassifier   # ML model

model = XGBClassifier(
    n_estimators=100,
    max_depth=4
)

model.fit(X_train, y_train)


# ================================
# 5. EVALUATE MODEL
# ================================

from sklearn.metrics import accuracy_score   # to calculate accuracy

preds = model.predict(X_test)
accuracy = accuracy_score(y_test, preds)

print("\nAccuracy:", accuracy)


# ================================
# 6. SAVE MODEL
# ================================

import joblib   # to save model

joblib.dump(model, "model.pkl")
joblib.dump(le, "encoder.pkl")

print("\nModel saved!")


# ================================
# 7. PREDICTION FUNCTION
# ================================

import numpy as np   # needed for input conversion

def predict_disease(data):
    
    input_data = np.array([[
        data["age"],
        data["gender"],
        data["bp"],
        data["sugar"],
        data["fever"],
        data["cough"],
        data["chest_pain"],
        data["fatigue"]
    ]])

    probs = model.predict_proba(input_data)[0]
    idx = probs.argmax()

    return {
        "disease": le.inverse_transform([idx])[0],
        "confidence": round(float(probs[idx]), 3)
    }


# ================================
# 8. TEST PREDICTION
# ================================

sample = {
    "age": 55,
    "gender": 0,
    "bp": 160,
    "sugar": 220,
    "fever": 0,
    "cough": 0,
    "chest_pain": 1,
    "fatigue": 1
}

print("\nPrediction:")
print(predict_disease(sample))

Dataset:
   age  gender   bp  sugar  fever  cough  chest_pain  fatigue          disease
0   45       0  140    180      0      0           1        1    Heart Disease
1   50       1  130    150      0      0           0        1         Diabetes
2   60       0  150    200      0      1           1        1     Hypertension
3   35       1  120     90      1      1           0        0        Pneumonia
4   29       0  110     85      0      0           0        0  Gastroenteritis

Encoded Labels: [2 0 3 4 1 2 0 3 4 1]

Accuracy: 0.5

Model saved!

Prediction:
{'disease': 'Heart Disease', 'confidence': 0.343}
